# Integrated Graph RAG Tutorial

This notebook is a guided walkthrough of the full pipeline behind the three demos:

- document ingestion from a URL
- preprocessing and chunking
- per-document Neo4j graph creation
- entity, relationship, and community summarization
- embeddings and retrieval
- schema-aware Text-to-Cypher generation
- orchestration with tool routing and repair

The tutorial is intentionally linear so readers can follow how raw text becomes a queryable graph application.

Practical note:
- Large documents can take time to process because extraction and orchestration may call the model multiple times.
- Output quality depends heavily on source text cleanliness, chunking, and preprocessing choices.
- For repeatable runs, always point the workflow at a clean Neo4j database to avoid duplicate nodes and stale state.


In [1]:
from pathlib import Path
import importlib
import hashlib
import json
import re
import sys
import requests
from tqdm import tqdm
from pydantic import BaseModel, Field, field_validator

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "runtime.py").exists():
    PROJECT_DIR = NOTEBOOK_DIR
elif (NOTEBOOK_DIR / "MyScripts" / "runtime.py").exists():
    PROJECT_DIR = NOTEBOOK_DIR / "MyScripts"
elif (NOTEBOOK_DIR.parent / "MyScripts" / "runtime.py").exists():
    PROJECT_DIR = NOTEBOOK_DIR.parent / "MyScripts"
else:
    raise RuntimeError("Could not locate the MyScripts project directory.")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import runtime as runtime_module
importlib.reload(runtime_module)
from runtime import chat, chunk_text, ensure_neo4j_driver, tool_choice, embed
from extraction_tools import (
    create_extraction_prompt as mature_create_extraction_prompt,
    parse_extraction_output as mature_parse_extraction_output,
    get_summarize_prompt as mature_get_summarize_prompt,
    calculate_communities as mature_calculate_communities,
    get_summarize_community_prompt as mature_get_summarize_community_prompt,
    extract_json as mature_extract_json,
    get_map_system_prompt as mature_get_map_system_prompt,
    get_reduce_system_prompt as mature_get_reduce_system_prompt,
    get_local_system_prompt as mature_get_local_system_prompt,
)
from cypher_generation import Text2Cypher as MatureText2Cypher, generate_prompt_sections as mature_generate_prompt_sections
from graph_orchestration_demo import (
    query_update_prompt as mature_query_update_prompt,
    answer_critique_prompt as mature_answer_critique_prompt,
    main_prompt as mature_main_prompt,
    repair_cypher as mature_repair_cypher,
)

driver = ensure_neo4j_driver()
SYSTEM_DATABASE = "system"


def database_exists(db_name: str) -> bool:
    records, _, _ = driver.execute_query(
        "SHOW DATABASES YIELD name RETURN name",
        database_=SYSTEM_DATABASE,
    )
    existing = {record["name"] for record in records}
    return db_name in existing


def create_database_if_needed(db_name: str) -> None:
    if not re.fullmatch(r"[a-z0-9-]+", db_name):
        raise ValueError(f"Unsafe database name: {db_name}")

    if database_exists(db_name):
        print(f"Database already exists: {db_name}")
        return

    driver.execute_query(
        f"CREATE DATABASE `{db_name}` IF NOT EXISTS",
        database_=SYSTEM_DATABASE,
    )
    print(f"Create command issued for database: {db_name}")


def wait_for_database(db_name: str, timeout_seconds: int = 120) -> None:
    import time

    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        records, _, _ = driver.execute_query(
            "SHOW DATABASES YIELD name, currentStatus RETURN name, currentStatus",
            database_=SYSTEM_DATABASE,
        )
        status = {record["name"]: record["currentStatus"] for record in records}
        if status.get(db_name) == "online":
            return
        time.sleep(2)
    raise TimeoutError(f"Database {db_name} did not become online within {timeout_seconds} seconds")


def slugify_database_name(text: str, prefix: str = "kg-rag-demo") -> str:
    text = re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")
    text = re.sub(r"-+", "-", text)
    if not text:
        text = "document"
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()[:8]
    return f"{prefix}-{text[:40]}-{digest}"


def reset_demo_graph(db_name: str) -> None:
    records, _, _ = driver.execute_query(
        "MATCH (n) RETURN count(n) AS node_count",
        database_=db_name,
    )
    node_count = records[0]["node_count"] if records else 0

    driver.execute_query(
        """
        MATCH (n)
        DETACH DELETE n
        """,
        database_=db_name,
    )

    print(f"Deleted {node_count} prior nodes in database: {db_name}")


def prepare_document_database(source_label: str, reset_if_exists: bool = True) -> str:
    db_name = slugify_database_name(source_label)
    create_database_if_needed(db_name)
    wait_for_database(db_name)
    if reset_if_exists:
        reset_demo_graph(db_name)
    return db_name


ACTIVE_DOC_DB = prepare_document_database("odyssey", reset_if_exists=True)
print(f"Using project directory: {PROJECT_DIR}")
print(f"Using document database: {ACTIVE_DOC_DB}")
print("Neo4j driver ready" if driver else "Neo4j driver not configured")

Database already exists: kg-rag-demo-odyssey-22fc7874
Deleted 207 prior nodes in database: kg-rag-demo-odyssey-22fc7874
Using project directory: /Users/qiaomu/weather/kg-rag/MyScripts
Using document database: kg-rag-demo-odyssey-22fc7874
Neo4j driver ready


## 1. Download the Source Document

We start from a public URL so the workflow is reproducible. This tutorial uses the Project Gutenberg text for the Odyssey.

We start from a public URL so the tutorial is reproducible.

In [2]:
SOURCE_URL = "https://www.gutenberg.org/cache/epub/1727/pg1727.txt"
response = requests.get(SOURCE_URL, timeout=30)
response.raise_for_status()
raw_text = response.text

print(f"Downloaded {len(raw_text):,} characters from {SOURCE_URL}")
print(raw_text[:800])

Downloaded 710,364 characters from https://www.gutenberg.org/cache/epub/1727/pg1727.txt
﻿The Project Gutenberg eBook of The Odyssey
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The Odyssey

Author: Homer

Translator: Samuel Butler


        
Release date: April 1, 1999 [eBook #1727]
                Most recently updated: December 2, 2023

Language: English

Other information and formats: www.gutenberg.org/ebooks/1727

Credits: Jim Tinsley and David


## 2. Preprocess and Chunk the Text

The raw book is cleaned only as much as needed for extraction. We split it into book-level sections and then into overlapping chunks so the extractor sees manageable pieces of text.

In [3]:
def split_into_books(text: str):
    body = text.split("PREFACE TO FIRST EDITION")[2]
    body = body.split("FOOTNOTES")[0].strip()
    return body.split("\nBOOK")[1:]

books = split_into_books(raw_text)
book_word_counts = [len(book.split()) for book in books]
print(f"Found {len(books)} books")
print(f"Average book length (words): {sum(book_word_counts) / len(book_word_counts):.1f}")
print(f"Shortest / longest book (words): {min(book_word_counts)} / {max(book_word_counts)}")
print("\nFirst book preview:\n")
print(books[0][:1200])

Found 24 books
Average book length (words): 4892.0
Shortest / longest book (words): 3357 / 8061

First book preview:

 I


THE GODS IN COUNCIL—MINERVA’S VISIT TO ITHACA—THE CHALLENGE FROM
TELEMACHUS TO THE SUITORS.


Tell me, O Muse, of that ingenious hero who travelled far and wide
after he had sacked the famous town of Troy. Many cities did he visit,
and many were the nations with whose manners and customs he was
acquainted; moreover he suffered much by sea while trying to save his
own life and bring his men safely home; but do what he might he could
not save his men, for they perished through their own sheer folly in
eating the cattle of the Sun-god Hyperion; so the god prevented them
from ever reaching home. Tell me, too, about all these things, oh
daughter of Jove, from whatsoever source you may know them.

So now all who escaped death in battle or by shipwreck had got safely
home except Ulysses, and he, though he was longing to return to his
wife and country, was detained by the 

In [4]:
chunked_books = [chunk_text(book, 1000, 40) for book in books]
print(f"First book split into {len(chunked_books[0])} chunks")
print("\nFirst chunk preview:\n")
print(chunked_books[0][0][:1200])

First book split into 22 chunks

First chunk preview:

I


THE GODS IN COUNCIL—MINERVA’S VISIT TO ITHACA—THE CHALLENGE FROM
TELEMACHUS TO THE SUITORS.


Tell me, O Muse, of that ingenious hero who travelled far and wide
after he had sacked the famous town of Troy. Many cities did he visit,
and many were the nations with whose manners and customs he was
acquainted; moreover he suffered much by sea while trying to save his
own life and bring his men safely home; but do what he might he could
not save his men, for they perished through their own sheer folly in
eating the cattle of the Sun-god Hyperion; so the god prevented them
from ever reaching home. Tell me, too, about all these things, oh
daughter of Jove, from whatsoever source you may know them.

So now all who escaped death in battle or by shipwreck had got safely
home except Ulysses, and he, though he was longing to return to his
wife and country, was detained by the goddess Calypso, who had got him
into a large cave and wanted to

## 3. Extract Entities and Relationships

This section shows the extraction loop in full. It defines the structured output format, validates the model response, and retries if the output is malformed.

In [5]:
ENTITY_TYPE_DEFINITIONS = {
    "AGENT": "An individual acting entity, such as a person, deity, ruler, named fictional being, or other named actor.",
    "ORGANIZATION": "A collective or institutional entity such as a company, government, army, religious order, team, or group.",
    "LOCATION": "A place or geographic/political region such as a city, country, sea, island, kingdom, or physical setting.",
    "EVENT": "A time-bounded occurrence such as a war, ceremony, ritual occurrence, meeting, battle, journey, or incident.",
    "DOCUMENT": "A text-bearing information object such as a book, report, contract, letter, paper, decree, or written record.",
    "ARTIFACT": "A man-made object, tool, weapon, vessel, or physical item that is not primarily a document or system.",
    "SYSTEM": "A technical, organizational, or functional system made of interacting components, such as a platform, pipeline, architecture, network, or process framework.",
    "CONCEPT": "An abstract idea, doctrine, role, title, policy, method, metric, belief, or category.",
    "OTHER": "Use only when none of the other entity types fit clearly.",
}

ENTITY_TYPES = list(ENTITY_TYPE_DEFINITIONS.keys())


def create_extraction_prompt(text: str) -> str:
    return mature_create_extraction_prompt(ENTITY_TYPES, text)


def parse_extraction_output(output_str: str):
    return mature_parse_extraction_output(output_str)


class EntityRecord(BaseModel):
    entity_name: str = Field(..., min_length=1)
    entity_type: str
    entity_description: str = Field(..., min_length=1)

    @field_validator("entity_name")
    @classmethod
    def normalize_entity_name(cls, v: str):
        return v.strip().upper()

    @field_validator("entity_type", mode="before")
    @classmethod
    def normalize_entity_type(cls, v):
        return str(v).strip().upper().replace(" ", "_").replace("-", "_")


class RelationshipRecord(BaseModel):
    source_entity: str = Field(..., min_length=1)
    target_entity: str = Field(..., min_length=1)
    relationship_description: str = Field(..., min_length=1)
    relationship_strength: float = Field(..., ge=0, le=10)

    @field_validator("source_entity", "target_entity")
    @classmethod
    def normalize_related_names(cls, v: str):
        return v.strip().upper()


def validate_records(raw_nodes, raw_relationships):
    valid_nodes, valid_relationships, rejected = [], [], []
    for node in raw_nodes:
        try:
            validated = EntityRecord.model_validate(node)
            valid_nodes.append(
                {
                    "entity_name": validated.entity_name,
                    "entity_type": validated.entity_type,
                    "entity_description": validated.entity_description,
                }
            )
        except Exception as e:
            rejected.append({"kind": "entity", "record": node, "error": str(e)})

    for rel in raw_relationships:
        try:
            validated = RelationshipRecord.model_validate(rel)
            valid_relationships.append(
                {
                    "source_entity": validated.source_entity,
                    "target_entity": validated.target_entity,
                    "relationship_description": validated.relationship_description,
                    "relationship_strength": validated.relationship_strength,
                }
            )
        except Exception as e:
            rejected.append({"kind": "relationship", "record": rel, "error": str(e)})

    return valid_nodes, valid_relationships, rejected


def extract_entities_from_chunk(text: str):
    prompt = create_extraction_prompt(text)
    for attempt in range(3):
        output = chat([{"role": "user", "content": prompt}], model="gpt-5-nano")
        raw_nodes, raw_relationships = parse_extraction_output(output)
        nodes, relationships, rejected = validate_records(raw_nodes, raw_relationships)
        if rejected and attempt < 2:
            prompt = prompt + "\n\nThe previous output had validation issues. Please repair the format and try again."
            continue
        return nodes, relationships
    return [], []

## 4. Graph Ingestion Per Document

The preview above only extracted one chunk. This section runs the same extraction pipeline over the full first book so the downstream summaries, communities, embeddings, and retrieval demos all have real data to work with.

Each processed document gets its own Neo4j database so the graph is isolated and easy to inspect.

In [6]:
import_nodes_query = """
MERGE (b:Document {id: $document_id})
MERGE (b)-[:HAS_CHUNK]->(c:__Chunk__ {id: $chunk_id})
SET c.text = $text
WITH c
UNWIND $data AS row
MERGE (n:__Entity__ {name: row.entity_name})
SET n:$(row.entity_type),
    n.description = coalesce(n.description, []) + [row.entity_description]
MERGE (c)-[:MENTIONS]->(n)
MERGE (c)-[:HAS_ENTITY]->(n)
"""

import_relationships_query = """
UNWIND $data AS row
MERGE (s:__Entity__ {name: row.source_entity})
MERGE (t:__Entity__ {name: row.target_entity})
MERGE (s)-[r:RELATIONSHIP {description: row.relationship_description, strength: row.relationship_strength}]->(t)
"""

FULL_INGEST_BOOK_INDEX = 0
FULL_INGEST_CHUNKS = chunked_books[FULL_INGEST_BOOK_INDEX]
document_id = f"book-{FULL_INGEST_BOOK_INDEX}"

document_chunk_result, _, _ = driver.execute_query(
    """
    MATCH (b:Document {id: $document_id})-[:HAS_CHUNK]->(c:__Chunk__)
    RETURN collect(toInteger(c.id)) AS chunk_ids, count(c) AS chunk_count
    """,
    {"document_id": document_id},
    database_=ACTIVE_DOC_DB,
)
document_chunk_ids = set(document_chunk_result[0]["chunk_ids"] or [])
document_chunk_count = document_chunk_result[0]["chunk_count"]

if document_chunk_count >= len(FULL_INGEST_CHUNKS):
    print(f"Reusing complete data in {ACTIVE_DOC_DB}; skipping document ingestion.")
else:
    chunks_to_process = [
        (chunk_i, chunk)
        for chunk_i, chunk in enumerate(FULL_INGEST_CHUNKS)
        if chunk_i not in document_chunk_ids
    ]

    print(f"Existing chunks in {ACTIVE_DOC_DB}: {len(document_chunk_ids)}")
    print(f"Chunks still to process: {len(chunks_to_process)}")

    for chunk_i, chunk in tqdm(chunks_to_process, desc="Ingesting full document"):
        nodes, relationships = extract_entities_from_chunk(chunk)
        driver.execute_query(
            import_nodes_query,
            {
                "document_id": document_id,
                "chunk_id": chunk_i,
                "text": chunk,
                "data": nodes,
            },
            database_=ACTIVE_DOC_DB,
        )
        driver.execute_query(
            import_relationships_query,
            {"data": relationships},
            database_=ACTIVE_DOC_DB,
        )

    print(f"Imported {len(chunks_to_process)} chunks into {ACTIVE_DOC_DB}")

Existing chunks in kg-rag-demo-odyssey-22fc7874: 0
Chunks still to process: 22


Ingesting full document:   0%|                           | 0/22 [00:00<?, ?it/s]

Ingesting full document:   5%|▊                 | 1/22 [02:03<43:10, 123.34s/it]

Ingesting full document:   9%|█▋                 | 2/22 [02:51<26:24, 79.25s/it]

Ingesting full document:  14%|██▌                | 3/22 [04:08<24:47, 78.30s/it]

...

Ingesting full document:  91%|████████████████▎ | 20/22 [26:58<01:57, 58.71s/it]

Ingesting full document:  95%|█████████████████▏| 21/22 [27:52<00:57, 57.38s/it]

Ingesting full document: 100%|██████████████████| 22/22 [28:32<00:00, 52.24s/it]

Ingesting full document: 100%|██████████████████| 22/22 [28:32<00:00, 77.86s/it]

Imported 22 chunks into kg-rag-demo-odyssey-22fc7874


## 5. Summarize Entities and Relationships

Entity summaries collapse repeated descriptions into a single readable summary. Relationship summaries do the same for repeated links between the same two entities.

In [7]:
def get_summarize_prompt(entity_name, description_list):
    return mature_get_summarize_prompt(entity_name, description_list)


def import_entity_summary(entity_information):
    driver.execute_query(
        """
        UNWIND $data AS row
        MATCH (e:__Entity__ {name: row.entity})
        SET e.summary = row.summary
        """,
        {"data": entity_information},
        database_=ACTIVE_DOC_DB,
    )

    driver.execute_query(
        """
        MATCH (e:__Entity__)
        WHERE size(e.description) = 1
        SET e.summary = e.description[0]
        """,
        database_=ACTIVE_DOC_DB,
    )


def import_rels_summary(rel_summaries):
    driver.execute_query(
        """
        UNWIND $data AS row
        MATCH (s:__Entity__ {name: row.source}), (t:__Entity__ {name: row.target})
        MERGE (s)-[r:SUMMARIZED_RELATIONSHIP]-(t)
        SET r.summary = row.summary
        """,
        {"data": rel_summaries},
        database_=ACTIVE_DOC_DB,
    )

    driver.execute_query(
        """
        MATCH (s:__Entity__)-[e:RELATIONSHIP]-(t:__Entity__)
        WHERE NOT (s)-[:SUMMARIZED_RELATIONSHIP]-(t)
        MERGE (s)-[r:SUMMARIZED_RELATIONSHIP]-(t)
        SET r.summary = e.description
        """,
        database_=ACTIVE_DOC_DB,
    )


entity_candidates, _, _ = driver.execute_query(
    """
    MATCH (e:__Entity__)
    WHERE e.summary IS NULL OR trim(e.summary) = ""
    AND size(coalesce(e.description, [])) > 1
    RETURN e.name AS entity_name, e.description AS description_list
    """,
    database_=ACTIVE_DOC_DB,
)

entity_summaries = []
for candidate in tqdm(entity_candidates, desc="Summarizing entities"):
    summary = chat([
        {"role": "user", "content": get_summarize_prompt(candidate["entity_name"], candidate["description_list"])}
    ], model="gpt-5-nano")
    entity_summaries.append({"entity": candidate["entity_name"], "summary": summary})

if entity_summaries:
    import_entity_summary(entity_summaries)
print(f"Summarized {len(entity_summaries)} entities")

relationship_candidates, _, _ = driver.execute_query(
    """
    MATCH (s:__Entity__)-[r:RELATIONSHIP]-(t:__Entity__)
    WHERE (r.summary IS NULL OR trim(r.summary) = "")
    WITH s.name AS source, t.name AS target,
         collect(r.description) AS description_list,
         count(*) AS count
    WHERE count > 1
    RETURN source, target, description_list
    """,
    database_=ACTIVE_DOC_DB,
)

relationship_summaries = []
for candidate in tqdm(relationship_candidates, desc="Summarizing relationships"):
    summary = chat([
        {"role": "user", "content": get_summarize_prompt(f"{candidate['source']} relationship to {candidate['target']}", candidate["description_list"])}
    ], model="gpt-5-nano")
    relationship_summaries.append({"source": candidate["source"], "target": candidate["target"], "summary": summary})

if relationship_summaries:
    import_rels_summary(relationship_summaries)
print(f"Summarized {len(relationship_summaries)} relationships")

Summarizing entities:   0%|                             | 0/148 [00:00<?, ?it/s]

Summarizing entities:   1%|▏                    | 1/148 [00:19<48:11, 19.67s/it]

Summarizing entities:   1%|▎                    | 2/148 [00:38<46:54, 19.27s/it]

Summarizing entities:   2%|▍                    | 3/148 [00:51<38:56, 16.12s/it]

Summarizing entities:   3%|▌                    | 4/148 [01:01<33:28, 13.95s/it]

...

Summarizing relationships:  94%|███████████████ | 34/36 [05:29<00:13,  6.57s/it]

Summarizing relationships:  97%|███████████████▌| 35/36 [05:33<00:05,  5.95s/it]

Summarizing relationships: 100%|████████████████| 36/36 [05:38<00:00,  5.62s/it]

Summarizing relationships: 100%|████████████████| 36/36 [05:38<00:00,  9.40s/it]

Summarized 36 relationships


## 6. Detect and Summarize Communities

Communities help the tutorial show how the graph can be organized into higher-level groups, then summarized into readable reports.

In [8]:
def calculate_communities(driver):
    return mature_calculate_communities(driver, database_=ACTIVE_DOC_DB)


def get_summarize_community_prompt(nodes, relationships):
    return mature_get_summarize_community_prompt(nodes, relationships)


def extract_json(text: str):
    return mature_extract_json(text)


def parse_community_report(text: str):
    cleaned = extract_json(text)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {
            "title": "Community summary",
            "summary": cleaned,
            "rating": 5.0,
            "rating_explanation": "Fallback parsing was used because the model did not return strict JSON.",
        }


community_info_query = """
MATCH (e:__Entity__)
WHERE e.louvain IS NOT NULL
WITH e.louvain AS louvain, collect(e) AS nodes
WHERE size(nodes) > 1
CALL apoc.path.subgraphAll(nodes[0], {whitelistNodes:nodes})
YIELD relationships
RETURN louvain AS communityId,
       [n in nodes | {id: n.name, description: n.summary, type: [el in labels(n) WHERE el <> '__Entity__'][0]}] AS nodes,
       [r in relationships | {start: startNode(r).name, type: type(r), end: endNode(r).name, description: r.description}] AS rels
"""

import_community_query = """
UNWIND $data AS row
MERGE (c:__Community__ {communityId: row.communityId})
SET c.title = row.community.title,
    c.summary = row.community.summary,
    c.rating = row.community.rating,
    c.rating_explanation = row.community.rating_explanation
WITH c, row
UNWIND row.nodes AS node
MERGE (n:__Entity__ {name: node})
MERGE (n)-[:IN_COMMUNITY]->(c)
"""

community_count_result, _, _ = driver.execute_query(
    """
    MATCH (c:__Community__)
    RETURN count(c) AS community_count
    """,
    database_=ACTIVE_DOC_DB,
)
community_count = community_count_result[0]["community_count"]

distinct_louvain_result, _, _ = driver.execute_query(
    """
    MATCH (e:__Entity__)
    WHERE e.louvain IS NOT NULL
    RETURN count(DISTINCT e.louvain) AS community_count
    """,
    database_=ACTIVE_DOC_DB,
)
expected_communities = distinct_louvain_result[0]["community_count"]

if community_count and community_count >= expected_communities:
    print(f"Community reports already exist in {ACTIVE_DOC_DB}; skipping community summarization.")
else:
    community_distribution = calculate_communities(driver)
    print(community_distribution)

    community_info, _, _ = driver.execute_query(community_info_query, database_=ACTIVE_DOC_DB)
    communities = []
    for community in tqdm(community_info, desc="Summarizing communities"):
        summary = chat([
            {"role": "user", "content": get_summarize_community_prompt(community["nodes"], community["rels"])}
        ], model="gpt-5-nano")
        communities.append({
            "community": parse_community_report(summary),
            "communityId": community["communityId"],
            "nodes": [el["id"] for el in community["nodes"]],
        })

    driver.execute_query(import_community_query, {"data": communities}, database_=ACTIVE_DOC_DB)
    print(f"Imported {len(communities)} community reports")

{'modularity': 0.6000840928819444, 'modularities': [0.5258653428819444, 0.5981716579861112, 0.6000840928819444], 'ranLevels': 3, 'communityCount': 18, 'communityDistribution': {'p1': 2, 'max': 39, 'p5': 2, 'p90': 23, 'p50': 4, 'p95': 39, 'p10': 2, 'p75': 6, 'p99': 39, 'p25': 2, 'min': 2, 'mean': 7.5, 'p999': 39}, 'preProcessingMillis': 0, 'computeMillis': 273, 'postProcessingMillis': 1, 'writeMillis': 16, 'nodePropertiesWritten': 135, 'configuration': {'maxIterations': 10, 'writeConcurrency': 4, 'seedProperty': None, 'consecutiveIds': False, 'maxLevels': 10, 'concurrency': 4, 'jobId': 'jid-8bd9f30a-7c76-4063-b0f4-9fa449a2832f', 'forceSeedOptimization': False, 'writeProperty': 'louvain', 'logProgress': True, 'includeIntermediateCommunities': False, 'nodeLabels': ['*'], 'sudo': False, 'relationshipTypes': ['*'], 'writeToResultStore': False, 'tolerance': 0.0001}}


Summarizing communities:   0%|                           | 0/18 [00:00<?, ?it/s]

Summarizing communities:   6%|█                  | 1/18 [00:20<05:55, 20.90s/it]

Summarizing communities:  11%|██                 | 2/18 [00:37<04:55, 18.46s/it]

Summarizing communities:  17%|███▏               | 3/18 [00:59<05:01, 20.09s/it]

...

Summarizing communities:  89%|████████████████  | 16/18 [04:55<00:38, 19.37s/it]

Summarizing communities:  94%|█████████████████ | 17/18 [05:18<00:20, 20.42s/it]

Summarizing communities: 100%|██████████████████| 18/18 [05:41<00:00, 21.18s/it]

Summarizing communities: 100%|██████████████████| 18/18 [05:41<00:00, 18.96s/it]

Imported 18 community reports


## 7. Add Embeddings and Build the Vector Index

The summaries become embeddings so the graph can support semantic retrieval.

In [9]:
from runtime import embed


def embed_texts(texts, model="text-embedding-3-small"):
    return embed(texts, model=model)

entities, _, _ = driver.execute_query(
    """
    MATCH (e:__Entity__)
    WHERE e.summary IS NOT NULL AND trim(e.summary) <> ""
    RETURN e.summary AS summary, e.name AS name
    """,
    database_=ACTIVE_DOC_DB,
)

entities_to_embed = []
for el in entities:
    entities_to_embed.append({"name": el["name"], "summary": el["summary"]})

embedded_count_result, _, _ = driver.execute_query(
    """
    MATCH (e:__Entity__)
    WHERE e.embedding IS NOT NULL
    RETURN count(e) AS embedding_count
    """,
    database_=ACTIVE_DOC_DB,
)
embedded_count = embedded_count_result[0]["embedding_count"]

if embedded_count >= len(entities_to_embed) and len(entities_to_embed) > 0:
    print(f"Embeddings already exist in {ACTIVE_DOC_DB}; skipping embedding step.")
else:
    data = [{"name": el["name"], "embedding": embed_texts([el["summary"]])[0]} for el in entities_to_embed]

    driver.execute_query(
        """
        UNWIND $data AS row
        MATCH (e:__Entity__ {name: row.name})
        CALL db.create.setNodeVectorProperty(e, 'embedding', row.embedding)
        """,
        {"data": data},
        database_=ACTIVE_DOC_DB,
    )

    driver.execute_query(
        """
        CREATE VECTOR INDEX entities IF NOT EXISTS
        FOR (n:__Entity__)
        ON (n.embedding)
        OPTIONS {indexConfig: {
          `vector.dimensions`: 1536,
          `vector.similarity_function`: 'cosine'
        }}
        """,
        database_=ACTIVE_DOC_DB,
    )

    print(f"Embedded {len(data)} entities and ensured vector index in {ACTIVE_DOC_DB}")

Embedded 148 entities and ensured vector index in kg-rag-demo-odyssey-22fc7874


## 8. Global and Local Retrieval

We now expose the two retrieval styles from the original extraction demo: a global community retriever and a local semantic retriever.

This section demonstrates both community-level retrieval and chunk-level semantic retrieval.

In [10]:
def concise_response_type(style: str = "paragraph") -> str:
    if style == "paragraph":
        return "one comprehensive paragraph, generally 4 ~ 10 sentences"
    if style == "bullets":
        return "exactly 3 bullet points, each bullet at most 20 words"
    if style == "qa":
        return "Answer: one direct answer in 2-3 sentences only"
    return "one short paragraph, maximum 4 sentences"


def global_retriever(query: str, rating_threshold: float = 5) -> str:
    community_data, _, _ = driver.execute_query(
        """
        MATCH (c:__Community__)
        WHERE c.rating >= $rating
        RETURN c.summary AS summary
        """,
        {"rating": rating_threshold},
        database_=ACTIVE_DOC_DB,
    )
    print(f"Got {len(community_data)} community summaries")
    intermediate_results = []
    for community in tqdm(community_data, desc="Processing communities"):
        intermediate_messages = [
            {"role": "system", "content": mature_get_map_system_prompt(community["summary"])},
            {"role": "user", "content": query},
        ]
        intermediate_results.append(chat(intermediate_messages, model="gpt-5-nano"))

    final_messages = [
        {
            "role": "system",
            "content": mature_get_reduce_system_prompt(
                intermediate_results,
                response_type=concise_response_type("paragraph"),
            ),
        },
        {"role": "user", "content": query},
    ]
    return chat(final_messages, model="gpt-5-nano")


local_search_query = """
CALL db.index.vector.queryNodes('entities', $k, $embedding)
YIELD node, score
WITH collect(node) as nodes
WITH collect {
    UNWIND nodes as n
    MATCH (n)<-[:HAS_ENTITY]-(c:__Chunk__)
    WITH c, count(distinct n) as freq
    RETURN c.text AS chunkText
    ORDER BY freq DESC
    LIMIT $topChunks
} AS text_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[:IN_COMMUNITY]->(c:__Community__)
    WITH c, coalesce(c.rating, 0) AS rating
    RETURN c.summary AS descriptionText
    ORDER BY rating DESC
    LIMIT $topCommunities
} AS report_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[r:SUMMARIZED_RELATIONSHIP]-(m)
    WHERE m IN nodes
    RETURN r.summary AS descriptionText
    ORDER BY coalesce(r.strength, 0) DESC
    LIMIT $topInsideRels
} as insideRels,
collect {
    UNWIND nodes as n
    RETURN n.summary AS descriptionText
} as entities
RETURN {Chunks: text_mapping, Reports: report_mapping,
       Relationships: insideRels,
       Entities: entities} AS text
"""


def local_search(query: str) -> str:
    context, _, _ = driver.execute_query(
        local_search_query,
        {"embedding": embed([query])[0], "topChunks": 3, "topCommunities": 3, "topInsideRels": 3, "k": 5},
        database_=ACTIVE_DOC_DB,
    )
    context_str = str(context[0]["text"])
    local_messages = [
        {
            "role": "system",
            "content": mature_get_local_system_prompt(
                context_str,
                response_type=concise_response_type("paragraph"),
            ),
        },
        {"role": "user", "content": query},
    ]
    return chat(local_messages, model="gpt-5-nano")


print(local_search("Who is Ulysses?"))
print(global_retriever("What does this book mainly talk about?"))

Ulysses, also known as Odysseus, is the legendary Greek hero and former king of Ithaca—the central figure of the Odyssey. Hermes warns that he must be allowed to depart and undertake the long voyage home, underscoring the divine involvement in his fate. He is celebrated as the brave father of Telemachus and a key reference point in the epic, with his arduous journey back to Ithaca forming the core arc of the narrative. In the text, there are moments that touch on his fate, including a line that “Ulysses is dead,” though the broader story presents him as alive and ultimately returning home. At a point in the tale, Minerva urges arming him with helmet, shield, and lances, signaling his rise to heroic leadership upon his return. Thus, Ulysses is the hero of cunning and endurance whose life centers on surviving ordeals and reclaiming his home and throne.
Got 16 community summaries


Processing communities:   0%|                            | 0/16 [00:00<?, ?it/s]

Processing communities:   6%|█▎                  | 1/16 [00:07<01:57,  7.82s/it]

Processing communities:  12%|██▌                 | 2/16 [00:15<01:45,  7.54s/it]

Processing communities:  19%|███▊                | 3/16 [00:21<01:31,  7.01s/it]

...

Processing communities:  88%|████████████████▋  | 14/16 [01:54<00:17,  8.67s/it]

Processing communities:  94%|█████████████████▊ | 15/16 [02:02<00:08,  8.64s/it]

Processing communities: 100%|███████████████████| 16/16 [02:08<00:00,  7.81s/it]

Processing communities: 100%|███████████████████| 16/16 [02:08<00:00,  8.04s/it]

From the aggregated analyst reports, there is insufficient information in the provided data tables to determine what the book mainly talks about. All reports state that the data tables do not contain any information about a book or its subject matter; they describe a community of entities, relationships, and claims instead. Consequently, the main topic cannot be determined from the available data. The consensus across the analyses is that the dataset lacks any entry for a book (title, content, or themes), so no meaningful summary of the book can be produced. Therefore, I cannot determine the book's main topic from the given data. [Data: Reports (1, 2, 3, 4, 5, +more)].


## 9. Inspect the Neo4j Schema

Text-to-Cypher works best when the prompt knows the available node labels, relationship types, and properties. This section shows how the schema is read directly from Neo4j.

In [11]:
NODE_PROPERTIES_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE NOT type = "RELATIONSHIP" AND elementType = "node"
WITH label AS nodeLabels, collect({property: property, type: type}) AS properties
RETURN {labels: nodeLabels, properties: properties} AS output
"""

REL_PROPERTIES_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE NOT type = "RELATIONSHIP" AND elementType = "relationship"
WITH label AS relType, collect({property: property, type: type}) AS properties
RETURN {type: relType, properties: properties} AS output
"""

REL_QUERY = """
CALL apoc.meta.data()
YIELD label, other, elementType, type, property
WHERE type = "RELATIONSHIP" AND elementType = "node"
UNWIND other AS other_node
RETURN {start: label, type: property, end: toString(other_node)} AS output
"""

def get_structured_schema(driver, mode="full_schema", database_name=None):
    node_labels_response = driver.execute_query(NODE_PROPERTIES_QUERY, database_=database_name or ACTIVE_DOC_DB)
    node_properties = [row.data()["output"] for row in node_labels_response.records]
    rel_properties_response = driver.execute_query(REL_PROPERTIES_QUERY, database_=database_name or ACTIVE_DOC_DB)
    rel_properties = [row.data()["output"] for row in rel_properties_response.records]
    rel_query_response = driver.execute_query(REL_QUERY, database_=database_name or ACTIVE_DOC_DB)
    relationships = [row.data()["output"] for row in rel_query_response.records]
    schema = {
        "node_props": {el["labels"]: el["properties"] for el in node_properties},
        "rel_props": {el["type"]: el["properties"] for el in rel_properties},
        "relationships": relationships,
    }
    if mode == "core_schema":
        summarized = [rel for rel in schema["relationships"] if rel["type"] == "SUMMARIZED_RELATIONSHIP"]
        related_labels = {rel["start"] for rel in summarized} | {rel["end"] for rel in summarized}
        schema = {
            "node_props": {label: props for label, props in schema["node_props"].items() if label in related_labels and label != "__Entity__"},
            "rel_props": {"SUMMARIZED_RELATIONSHIP": schema["rel_props"].get("SUMMARIZED_RELATIONSHIP", [])},
            "relationships": [rel for rel in summarized if rel["start"] in related_labels and rel["end"] in related_labels],
        }
    return schema

def get_schema(driver, mode="full_schema", database_name=None):
    structured = get_structured_schema(driver, mode=mode, database_name=database_name)
    parts = ["Node properties:"]
    for label, props in structured["node_props"].items():
        prop_text = ", ".join(f"{p['property']}: {p['type']}" for p in props)
        parts.append(label + " {" + prop_text + "}")
    parts.append("Relationship properties:")
    for rel_type, props in structured["rel_props"].items():
        prop_text = ", ".join(f"{p['property']}: {p['type']}" for p in props)
        parts.append(rel_type + " {" + prop_text + "}")
    parts.append("The relationships:")
    for rel in structured["relationships"]:
        parts.append(f"(:{rel['start']})-[:{rel['type']}]->(:{rel['end']})")
    return "\n".join(parts)

schema_string = get_schema(driver, mode="core_schema", database_name=ACTIVE_DOC_DB)
print(schema_string)

Node properties:
AGENT {name: STRING, description: LIST, summary: STRING, louvain: INTEGER, embedding: LIST}
LOCATION {name: STRING, description: LIST, summary: STRING, louvain: INTEGER, embedding: LIST}
CONCEPT {name: STRING, description: LIST, summary: STRING, louvain: INTEGER, embedding: LIST}
ARTIFACT {name: STRING, description: LIST, summary: STRING, louvain: INTEGER, embedding: LIST}
EVENT {name: STRING, description: LIST, summary: STRING, louvain: INTEGER, embedding: LIST}
OTHER {name: STRING, description: LIST, summary: STRING, louvain: INTEGER, embedding: LIST}
ORGANIZATION {name: STRING, description: LIST, summary: STRING, louvain: INTEGER, embedding: LIST}
Relationship properties:
SUMMARIZED_RELATIONSHIP {summary: STRING}
The relationships:
(:__Entity__)-[:SUMMARIZED_RELATIONSHIP]->(:__Entity__)
(:__Entity__)-[:SUMMARIZED_RELATIONSHIP]->(:AGENT)
(:__Entity__)-[:SUMMARIZED_RELATIONSHIP]->(:CONCEPT)
(:__Entity__)-[:SUMMARIZED_RELATIONSHIP]->(:LOCATION)
(:__Entity__)-[:SUMMARIZ

## 10. Build Prompt Sections for Text-to-Cypher

We now turn the schema into user-facing terminology and examples. This step teaches the model how the graph terminology maps to natural language.

In [12]:
sections = mature_generate_prompt_sections(driver, database_=ACTIVE_DOC_DB)
print(sections["terminology_string"])
print("\n--- Examples ---\n")
print(sections["examples_string"])


Characters: When a user asks about agent, character, person, figure, hero, myth figure, titan, god, ruler, or leader, they are referring to a node with the label 'AGENT'.
Locations: When a user asks about location, place, site, spot, area, region, landmark, geography, locality, or space, they are referring to a node with the label 'LOCATION'.
Concepts: When a user asks about concept, idea, theme, notion, motif, topic, symbol, renown, or phenomenon, they are referring to a node with the label 'CONCEPT'.
Artifacts: When a user asks about artifact, object, item, tool, vessel, utensil, container, diningware, or equipment, they are referring to a node with the label 'ARTIFACT'.
Events: When a user asks about event, occurrence, happening, incident, journey, voyage, visit, ceremony, wedding, or festival, they are referring to a node with the label 'EVENT'.
Others: When a user asks about other, misc, miscellaneous, oddity, outlier, uncategorized, leftover, detail, or item, they are referring t

## 11. Generate Cypher From a Natural-Language Question

The `Text2Cypher` wrapper combines the schema, terminology, examples, and user question into one prompt.

In [13]:
question = "How is ULYSSES related to NEPTUNE?"
t2c = MatureText2Cypher(driver=driver, schema_mode="core_schema", database_=ACTIVE_DOC_DB)
t2c.set_prompt_section("question", question)
t2c.set_prompt_section("terminology", sections["terminology_string"])
t2c.set_prompt_section("examples", sections["examples_string"])

cypher = t2c.generate_cypher()
print(cypher)

try:
    records, _, _ = driver.execute_query(cypher, database_=ACTIVE_DOC_DB)
    print([record.data() for record in records[:5]])
except Exception as exc:
    print(f"Cypher execution failed: {exc}")

MATCH (a:AGENT {name: 'ULYSSES'})-[r:SUMMARIZED_RELATIONSHIP]-(n)
WHERE n.name = 'NEPTUNE'
RETURN a.name AS ULYSSES, n.name AS NEPTUNE, type(r) AS relationship, r.summary AS summary
[{'ULYSSES': 'ULYSSES', 'NEPTUNE': 'NEPTUNE', 'relationship': 'SUMMARIZED_RELATIONSHIP', 'summary': 'Neptune is furious with Ulysses for having blinded Polyphemus and may be pacified only if all are of one mind'}]


## 12. Orchestrate the Full Question-Answering Loop

The final section puts the pieces together. The router can rewrite the question, choose the best tool, repair bad Cypher, and synthesize the final answer.
This section shows both orchestration modes directly.
The comparison section below remains optional.

In [19]:
import importlib
import graph_orchestration_demo as orchestration_module
importlib.reload(orchestration_module)
orchestration_main = orchestration_module.main
import pandas as pd

question = "How is ULYSSES related to NEPTUNE?"

print("Running orchestration demo: agentic mode")
response_agentic = orchestration_main(question, database_=ACTIVE_DOC_DB, mode="agentic")
print(response_agentic)

print("\nRunning orchestration demo: stable mode")
response_stable = orchestration_main(question, database_=ACTIVE_DOC_DB, mode="stable")
print(response_stable)

orchestration_rows = [
    {
        "method": "orchestration_agentic",
        "best_for": "The agentic orchestration loop",
        "question": question,
        "mode": "agentic",
        "result": response_agentic,
    },
    {
        "method": "orchestration_stable",
        "best_for": "The stable orchestration mode",
        "question": question,
        "mode": "stable",
        "result": response_stable,
    },
]

orchestration_df = pd.DataFrame(
    {
        "mode": row["mode"],
        "label": row["method"],
        "response_preview": str(row["result"])[:600],
    }
    for row in orchestration_rows
)
print("\nOrchestration summary:\n")
print(orchestration_df.to_string(index=False))


Running orchestration demo: agentic mode


From the available data, there is no documented relationship between ULYSSES and Neptune, and no Neptune flyby of ULYSSES is recorded.

What is missing:
- A complete, authoritative dataset or source that lists ULYSSES mission events and any interactions with Neptune (if any).
- Confirmation of whether any non-flyby interactions (e.g., observations of Neptune, gravitational studies) are recorded for ULYSSES in other data sources.

If you want, I can try to retrieve additional data from other sources or specify what exact type of relationship you want to check (flyby, flyby-free encounters, observations, etc.).

Running orchestration demo: stable mode


In the knowledge graph, ULYSSES and NEPTUNE are connected by a SUMMARIZED_RELATIONSHIP: Neptune is furious with ULYSSES for having blinded Polyphemus and may be pacified only if all are of one mind.

Orchestration summary:

   mode                 label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               response_preview
agentic orchestration_agentic From the available data, there is no documented relationship between ULYSSES and Neptune, and no Neptune flyb

## 13. Compare the Question-Answering Methods

This graph RAG stack includes multiple question-answering strategies. They solve different classes of questions, so a good tutorial should show the tradeoffs explicitly.

- `entity_info_by_name` is best for one named entity and its direct neighborhood.
- `local_search` is best when you want focused evidence from chunks, entity summaries, and nearby community context.
- `global_retriever` is best for broad questions that need community-level synthesis.
- `Text2Cypher` is best when the question needs precise graph structure, filters, joins, counts, or paths.
- `orchestration_main` is the router that chooses between these methods automatically.

The table and example outputs below compare those methods on representative questions.
Orchestration is already shown in the previous section, so this comparison stays focused on the other QA methods.


In [20]:
from graph_tools import entity_info_by_name
import pandas as pd

comparison_question = "How is ULYSSES related to NEPTUNE?"


def compact_entity_info(result):
    if not result:
        return None
    first = result[0]
    entity = first.get("entity", {})
    return {
        "labels": first.get("entity_labels", []),
        "name": entity.get("name"),
        "summary": str(entity.get("summary", ""))[:500],
        "relationship_type": first.get("relationship_type"),
        "direction": first.get("direction"),
        "related_entity_labels": first.get("related_entity_labels", []),
    }


def summarize_result(result, max_chars: int = 700):
    if isinstance(result, str):
        return result[:max_chars]
    if isinstance(result, dict):
        if "cypher" in result:
            return {
                "cypher": result["cypher"],
                "row_count": len(result.get("rows", [])),
                "rows_preview": result.get("rows", [])[:2],
            }
        return result
    if isinstance(result, list):
        return compact_entity_info(result)
    return str(result)[:max_chars]


comparison_rows = [
    {
        "method": "entity_info_by_name",
        "best_for": "A single named entity and its direct neighborhood",
        "question": "ULYSSES",
        "result": summarize_result(entity_info_by_name("ULYSSES", database_=ACTIVE_DOC_DB)),
    },
    {
        "method": "local_search",
        "best_for": "Targeted evidence from chunks, summaries, and nearby context",
        "question": "Who is Ulysses?",
        "result": summarize_result(local_search("Who is Ulysses?")),
    },
    {
        "method": "global_retriever",
        "best_for": "A broad, community-level answer",
        "question": "What does this book mainly talk about?",
        "result": summarize_result(global_retriever("What does this book mainly talk about?")),
    },
    {
        "method": "Text2Cypher",
        "best_for": "A structured graph query with paths, joins, counts, or filters",
        "question": comparison_question,
        "result": None,
    },
]

sections = mature_generate_prompt_sections(driver, database_=ACTIVE_DOC_DB)
t2c = MatureText2Cypher(driver=driver, schema_mode="core_schema", database_=ACTIVE_DOC_DB)
t2c.set_prompt_section("question", comparison_question)
t2c.set_prompt_section("terminology", sections["terminology_string"])
t2c.set_prompt_section("examples", sections["examples_string"])
cypher = t2c.generate_cypher()
cypher_records, _, _ = driver.execute_query(cypher, database_=ACTIVE_DOC_DB)
comparison_rows[3]["result"] = summarize_result({
    "cypher": cypher,
    "rows": [record.data() for record in cypher_records[:5]],
})

comparison_rows.extend([
    {
        "method": row["method"],
        "best_for": row["best_for"],
        "question": row["question"],
        "result": summarize_result(row["result"]),
    }
    for row in orchestration_rows
])

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

for row in comparison_rows:
    print()
    print(f"### {row['method']}")
    print(f"Best for: {row['best_for']}")
    print(f"Question: {row['question']}")
    print(json.dumps(row['result'], ensure_ascii=False, indent=2, default=str)[:1800])


Got 16 community summaries


Processing communities:   0%|                            | 0/16 [00:00<?, ?it/s]

Processing communities:   6%|█▎                  | 1/16 [00:05<01:24,  5.60s/it]

Processing communities:  12%|██▌                 | 2/16 [00:14<01:42,  7.33s/it]

Processing communities:  19%|███▊                | 3/16 [00:21<01:38,  7.55s/it]

...

Processing communities:  88%|████████████████▋  | 14/16 [01:49<00:16,  8.39s/it]

Processing communities:  94%|█████████████████▊ | 15/16 [01:56<00:08,  8.06s/it]

Processing communities: 100%|███████████████████| 16/16 [02:04<00:00,  8.06s/it]

Processing communities: 100%|███████████████████| 16/16 [02:04<00:00,  7.79s/it]

               method                                                       best_for                               question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       result
  entity_info_by_name              A single named entity and its direct neighborhood                                ULYSSES                                                    

## 14. Wrap-Up

The tutorial is structured so readers can follow the data as it moves through the system: raw URL text, cleaned chunks, extracted graph records, schema-aware prompt building, Cypher generation, and final orchestration.

By this point the notebook has exercised every major capability from the source demos in a single workflow.